In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

# warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

2025-04-06 16:07:47,115 - root - ERROR - Error writing configs: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'
Traceback (most recent call last):
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/client/config/config_helpers.py", line 876, in save_to_yml
    with open(yml_path, "w", encoding="utf-8") as outfile:
FileNotFoundError: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'


In [3]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal
from controllers.directional_trading.pz_simple_trend import PZSimpleDirectionalControllerConfig


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "5m"
backtesting_resolution = "5m"

# Indicator Values
ema_fast: int = 10
ema_medium: int = 20
ema_slow: int = 50
#####

total_amount_quote = 1000
max_executors_per_side = 2
time_limit = 60 * 60 * 12 * 9999 # disable time limit
cooldown_time = 1 #60 * 15
take_profit = 1.0 # 100%, -> Disable Take profit, let the trailing do it's job
stop_loss = 0.01
trailing_stop_activation_price = 0.015 #0.015
trailing_stop_trailing_delta = 0.005
sl_natr_factor = 1.0
natr_length = 14
ts_activation_natr_factor = 5.0
ts_delta_natr_factor = 5.5


# Creating the instance of the configuration and the controller
config = PZSimpleDirectionalControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    natr_length = natr_length,
    sl_natr_factor=sl_natr_factor,
    ts_activation_natr_factor = ts_activation_natr_factor,
    ts_delta_natr_factor = ts_delta_natr_factor,
    ema_fast=ema_fast,
    ema_medium=ema_medium,
    ema_slow=ema_slow,
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 1, 1).timestamp())
end = int(datetime.datetime(2025, 3, 31).timestamp())

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution)

2025-04-06 16:07:48,289 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x763eeac91f90>
2025-04-06 16:07:48,290 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x763f6bca0fa0>, 72409.26792126)])']
connector: <aiohttp.connector.TCPConnector object at 0x763f6b213220>
2025-04-06 16:07:49,316 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x763ec369e140>
2025-04-06 16:07:49,317 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x763ec3690e80>, 72410.294982987)])']
connector: <aiohttp.connector.TCPConnector object at 0x763ec369e170>
2025-04-06 16:07:54,764 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x763ec369d660>
2025-04-06 16:07:54,765 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.clie

In [5]:

# candles_df = backtesting_result.processed_data
# candles_df

In [6]:
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data



ema_fast_key = f"EMA_{ema_fast}"
ema_medium_key = f"EMA_{ema_medium}"
ema_slow_key = f"EMA_{ema_slow}"


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_medium_key],
                         line=dict(color='#FFA500', width=2),
                         name='Slow HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow_key],
                         line=dict(color='#FFFFFF', width=2),
                         name='Slow EMA'))



Net PNL: $0.51 (0.05%) | Max Drawdown: $-144.44 (-14.49%)
Total Volume ($): 312500.00 | Sharpe Ratio: -0.30 | Profit Factor: 1.00
Total Executors: 467 | Accuracy Long: 0.20 | Accuracy Short: 0.25
Close Types: Take Profit: 0 | Stop Loss: 145 | Time Limit: 0 |
             Trailing Stop: 13 | Early Stop: 309



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grab processed data
candles_df = backtesting_result.processed_data

# Create subplots layout
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.60, 0.15, 0.15, 0.1],
    vertical_spacing=0.02,
    subplot_titles=["Price + Indicators", "Condition", "Crossover"]
)

# Row 1: Close price
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df["close"],
    line=dict(color='#FFFFFF', width=2), name='Close'), row=1, col=1)


ema_fast_key = f"EMA_{ema_fast}"
ema_medium_key = f"EMA_{ema_medium}"
ema_slow_key = f"EMA_{ema_slow}"


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_medium_key],
                         line=dict(color='#FFA500', width=2),
                         name='Slow HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow_key],
                         line=dict(color='#FFFFFF', width=2),
                         name='Slow EMA'))

# Row 4: Signal values as line + markers
# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df["short_condition"],
#     mode='lines+markers',
#     line=dict(color='cyan', width=2),
#     name='short_condition',
#     marker=dict(size=6)
# ), row=2, col=1)

# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df["bearish_crossover"],
#     mode='lines+markers',
#     line=dict(color='cyan', width=2),
#     name='bearish_crossover',
#     marker=dict(size=6)
# ), row=3, col=1)
fig.add_trace(go.Scatter(
    x=candles_df.index,
    y=candles_df["signal"],
    mode='lines+markers',
    line=dict(color='cyan', width=2),
    name='signal',
    marker=dict(size=6)
), row=2, col=1)


# Final layout
fig.update_layout(
    height=700,
    title="Backtest with Candles, HMAs, RSI, and StochRSI",
    showlegend=True,
    template="plotly_dark"
)

fig.show()



KeyError: 'short_condition'

In [ ]:
# # 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,EriwCKqSwgZeUcBybi1S8Qhx1zLWF22S1VRa8WEci5Ct,1738379400,position_executor,1738380000,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'EriwCKqSwgZeUcBybi1S8Qhx1zLWF22S1VRa8W...,-0.0017200716845878309679829243350468459539115...,-0.8600358422939153868469475128222256898880004...,0.29999999999999993338661852249060757458209991...,499.99999999999994315658113919198513031005859375,False,True,"{'close_price': 1.7876, 'level_id': None, 'sid...",None,SELL
1,FhEQi93kzA14aeC1NhiQknhAx11FTkLdL9uniNPGZSW1,1738385700,position_executor,1738386600,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'FhEQi93kzA14aeC1NhiQknhAx11FTkLdL9uniN...,-0.0041413153457000133339516168007321539334952...,-2.0706576728500065698312937456648796796798706...,0.29999999999999993338661852249060757458209991...,499.99999999999994315658113919198513031005859375,False,True,"{'close_price': 1.7853, 'level_id': None, 'sid...",None,SELL
2,4uykEgjezLp7tttVsK9UuZbE15uzfCVmPVAC4KLYYMBo,1738390200,position_executor,1738406400,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '4uykEgjezLp7tttVsK9UuZbE15uzfCVmPVAC4K...,0.02611090367204642673226722138224431546404957...,13.0554518360232147955457548960112035274505615...,0.29999999999999998889776975374843459576368331...,500.00000000000005684341886080801486968994140625,False,True,"{'close_price': 1.7308, 'level_id': None, 'sid...",None,SELL
3,HN2HXZnGvpydqZiUTaJTWBhTiipefUJwXh7MXYP8pKTM,1738407600,position_executor,1738408800,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': 'HN2HXZnGvpydqZiUTaJTWBhTiipefUJwXh7MXY...,-0.0086771689232377317846367503761939588002860...,-4.3385844616188657951738605333957821130752563...,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.7348, 'level_id': None, 'sid...",None,SELL
4,6BCgy6AzhvAoa7t9uDDWj5x2ZFM42kz7SVagetLx6WH3,1738417500,position_executor,1738430400,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '6BCgy6AzhvAoa7t9uDDWj5x2ZFM42kz7SVaget...,0.00655449046067906396229041376955137820914387...,3.27724523033953163420051168941427022218704223...,0.29999999999999993338661852249060757458209991...,499.99999999999994315658113919198513031005859375,False,True,"{'close_price': 1.7069, 'level_id': None, 'sid...",None,SELL
5,8whfnHB588BdQ9mgBjhVryzdZKrLjZqnuSBfuF1my6zD,1738431000,position_executor,1738457100,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '8whfnHB588BdQ9mgBjhVryzdZKrLjZqnuSBfuF...,0.03798842936255468483164321469303104095160961...,18.9942146812773415831543388776481151580810546875,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.6319, 'level_id': None, 'sid...",None,SELL
6,5Wf5bdnFgx7pxntbmsTTmW26F97J1EBQJ3fPLk8766Jr,1738446900,position_executor,1738457100,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '5Wf5bdnFgx7pxntbmsTTmW26F97J1EBQJ3fPLk...,0.00397484445528821853099721295166091294959187...,1.98742222764410958468772605556296184659004211...,0.29999999999999998889776975374843459576368331...,500.00000000000005684341886080801486968994140625,False,True,"{'close_price': 1.6319, 'level_id': None, 'sid...",None,SELL
7,3Wx6QCnb7W9JEbtDA1pWpv9YSwPffeAJsoXfmtd2TdyA,1738461900,position_executor,1738474800,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '3Wx6QCnb7W9JEbtDA1pWpv9YSwPffeAJsoXfmt...,0.01819676003215191587014665230981336208060383...,9.09838001607595714403942110948264598846435546875,0.29999999999999998889776975374843459576368331...,500,False,True,"{'close_price': 1.5869, 'level_id': None, 'sid...",None,SELL
8,25J17BMzXoJ86GnnGr24pjQhuW8dVY2vPpH7mYFRNzBy,1738491300,position_executor,1738494300,CloseType.EARLY_STOP,RunnableStatus.TERMINATED,{'id': '25J17BMzXoJ86GnnGr24pjQhuW8dVY2vPpH7mY...,0.00009182389937116900140851027956045982136856...,0.04591194968558449940321253279762458987534046...,0.2999999999999999888977697537

### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [ ]:
# import plotly.express as px

# # Create a new column for profitability
# executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# # Create the scatter plot
# fig = px.scatter(
#     executors_df,
#     x="timestamp",
#     y='net_pnl_quote',
#     title='PNL per Trade',
#     color='profitable',
#     color_discrete_map={True: 'green', False: 'red'},
#     labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
#     hover_data=['filled_amount_quote', 'side']
# )

# # Customize the layout
# fig.update_layout(
#     xaxis_title="Timestamp",
#     yaxis_title="Net PNL (Quote)",
#     legend_title="Profitable",
#     font=dict(size=12, color="white"),
#     showlegend=False,
#     plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
#     paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
#     xaxis=dict(gridcolor="gray"),
#     yaxis=dict(gridcolor="gray")
# )

# # Add a horizontal line at y=0 to clearly separate profits and losses
# fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# # Show the plot
# fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [ ]:
# fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
# fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT